# Review Full Dataset Data Sources

Use this notebook to formally inspect the external data sources for the full `fake_mantis_invest` universe. The goal here is not only to fetch the data, but to dereference URLs and filing paths, extract preview text, and judge how useful each source will be for the long-term product.


## Scope

- Build the full ticker universe from the fake Mantis dataset.
- Fetch official SEC company mappings and filing metadata.
- Build full-document URLs for latest 10-K / 10-Q filings.
- Inflate a sample of those filing URLs into extracted text previews.
- Fetch macro context from FRED.
- Fetch company profile enrichment from FMP.
- Fetch broad news metadata from GDELT.
- Inflate a sample of news URLs into extracted text previews.
- Create a source-review table describing what each source is useful for.

### Helpful Short Forms

- `SEC` = **U.S. Securities and Exchange Commission**
- `CIK` = **Central Index Key**, the SEC company identifier
- `FRED` = **Federal Reserve Economic Data**
- `GDELT` = **Global Database of Events, Language, and Tone**
- `FMP` = **Financial Modeling Prep**
- `10-K` = annual filing
- `10-Q` = quarterly filing
- `8-K` = current event filing
- `MD&A` = **Management's Discussion and Analysis**


In [15]:
from __future__ import annotations

import gzip
import json
import os
import re
import time
from pathlib import Path
from urllib.error import HTTPError
from urllib.parse import quote
from urllib.request import Request, urlopen

import pandas as pd
from dotenv import load_dotenv

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'data_pipeline' else Path.cwd().resolve()
DATA_DIR = ROOT / 'data'
RAW_DIR = DATA_DIR / 'raw'
EXTERNAL_DIR = DATA_DIR / 'external'
INTERIM_DIR = DATA_DIR / 'interim'
REVIEW_DIR = INTERIM_DIR / 'source_reviews'
load_dotenv(ROOT / '.env')

for path in [
    EXTERNAL_DIR / 'sec',
    EXTERNAL_DIR / 'fred',
    EXTERNAL_DIR / 'gdelt',
    EXTERNAL_DIR / 'gdelt' / 'cache',
    EXTERNAL_DIR / 'fmp',
    REVIEW_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

SEC_HEADERS = {
    'User-Agent': 'portfolio-analyzer research contact@example.com',
    'Accept-Encoding': 'gzip, deflate',
}

MAX_SEC_DOCS_TO_INFLATE = 10
MAX_GDELT_ARTICLES_PER_TICKER = 3
MAX_NEWS_URLS_TO_INFLATE = 10
GDELT_REQUEST_DELAY_SECONDS = 5.2
GDELT_SOURCE_COUNTRY = 'US'
GDELT_SOURCE_LANGUAGE = 'English'

def read_response_bytes(response) -> bytes:
    payload = response.read()
    encoding = (response.headers.get('Content-Encoding') or '').lower()
    if 'gzip' in encoding:
        payload = gzip.decompress(payload)
    return payload

def fetch_url(url: str, headers: dict[str, str] | None = None, *, retries: int = 4, base_delay: float = 1.5) -> bytes:
    merged_headers = headers or {}
    last_error = None
    for attempt in range(retries):
        request = Request(url, headers=merged_headers)
        try:
            with urlopen(request) as response:
                return read_response_bytes(response)
        except HTTPError as error:
            last_error = error
            if error.code != 429 or attempt == retries - 1:
                raise
            retry_after = error.headers.get('Retry-After')
            delay = float(retry_after) if retry_after else base_delay * (2 ** attempt)
            print(f'Rate limited for {url}. Sleeping {delay:.1f}s before retry {attempt + 2}/{retries}.')
            time.sleep(delay)
    raise last_error if last_error else RuntimeError('Request failed without a captured HTTPError')

def fetch_json(url: str, headers: dict[str, str] | None = None) -> dict | list:
    return json.loads(fetch_url(url, headers=headers).decode('utf-8'))

def fetch_text(url: str, headers: dict[str, str] | None = None) -> str:
    return fetch_url(url, headers=headers).decode('utf-8', errors='ignore')


def load_cached_json(path: Path) -> dict | list | None:
    if not path.exists():
        return None
    return json.loads(path.read_text())

def write_cached_json(path: Path, payload: dict | list) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload))

def html_to_text(html: str) -> str:
    no_script = re.sub(r'<(script|style)[^>]*>.*?</\\1>', ' ', html, flags=re.I | re.S)
    no_tags = re.sub(r'<[^>]+>', ' ', no_script)
    return re.sub(r'\s+', ' ', no_tags).strip()

def clean_filing_text(text: str) -> str:
    clean = text
    clean = re.sub(r'\b(?:us-gaap|dei|xbrli|xbrldi|srt|link|iso4217):[A-Za-z0-9_]+\b', ' ', clean)
    clean = re.sub(r'https?://\S+', ' ', clean)
    clean = re.sub(r'\b\d{10}[- ]?\d{2}[- ]?\d{6}\b', ' ', clean)
    clean = re.sub(r'\bCIK\b|\bExact name of registrant as specified in its charter\b', ' ', clean, flags=re.I)
    clean = re.sub(r'\bPage\s+\d+\b', ' ', clean, flags=re.I)
    clean = re.sub(r'\bTable of Contents\b', ' ', clean, flags=re.I)
    clean = re.sub(r'[_•·]+', ' ', clean)
    clean = re.sub(r'\s+', ' ', clean).strip()
    return clean


def clean_article_text(text: str) -> str:
    clean = text
    clean = re.sub(r'https?://\S+', ' ', clean)
    clean = re.sub(r'\b(?:cookie|privacy policy|terms of service|advertisement|subscribe|sign up|newsletter)\b', ' ', clean, flags=re.I)
    clean = re.sub(r'\s+', ' ', clean).strip()
    return clean

def simple_chunk(text: str, chunk_size: int = 1800) -> list[str]:
    clean = re.sub(r'\s+', ' ', text).strip()
    return [clean[i:i + chunk_size] for i in range(0, len(clean), chunk_size) if clean[i:i + chunk_size].strip()]

def find_item_heading(text: str, candidates: list[str]) -> int | None:
    lowered = text.lower()
    matches = []
    for candidate in candidates:
        idx = lowered.find(candidate.lower())
        if idx >= 0:
            matches.append(idx)
    return min(matches) if matches else None

def extract_section_preview(text: str, start_candidates: list[str], *, preview_chars: int = 2200) -> str:
    start = find_item_heading(text, start_candidates)
    if start is None:
        return ''
    next_match = re.search(r'item\s+\d+[a-z]?\.?\s', text[start + 20:], flags=re.I)
    end = start + 20 + next_match.start() if next_match else min(len(text), start + preview_chars)
    snippet = text[start:end]
    return snippet[:preview_chars].strip()

def get_filing_section_previews(clean_text: str, form_type: str) -> dict[str, str]:
    if form_type == "10-K":
        risk = extract_section_preview(clean_text, ["item 1a. risk factors", "item 1a risk factors", "risk factors"])
        mda = extract_section_preview(clean_text, ["item 7. management\'s discussion and analysis", "item 7 management\'s discussion and analysis", "management\'s discussion and analysis"])
        business = extract_section_preview(clean_text, ["item 1. business", "item 1 business"])
    else:
        risk = extract_section_preview(clean_text, ["item 1a. risk factors", "item 1a risk factors", "risk factors"])
        mda = extract_section_preview(clean_text, ["item 2. management\'s discussion and analysis", "item 2 management\'s discussion and analysis", "management\'s discussion and analysis"])
        business = extract_section_preview(clean_text, ["business"])
    return {
        'risk_factors_preview': risk,
        'mda_preview': mda,
        'business_preview': business,
    }

def build_primary_doc_url(cik: str, accession_number: str, primary_document: str) -> str:
    accession = accession_number.replace('-', '')
    return f'https://www.sec.gov/Archives/edgar/data/{int(cik)}/{accession}/{primary_document}'


def classify_fact_tag(fact_tag: str) -> dict[str, str]:
    tag = (fact_tag or '').strip()
    lower = tag.lower()

    exact_map = {
        'revenues': ('income_statement', 'revenue', 'revenue', 'core', 'use_now'),
        'salesrevenuenet': ('income_statement', 'revenue', 'revenue', 'core', 'use_now'),
        'costofrevenue': ('income_statement', 'costs', 'cost_of_revenue', 'core', 'use_now'),
        'costofgoodsold': ('income_statement', 'costs', 'cost_of_goods_sold', 'core', 'use_now'),
        'grossprofit': ('income_statement', 'profitability', 'gross_profit', 'core', 'use_now'),
        'researchanddevelopmentexpense': ('income_statement', 'expenses', 'research_and_development', 'secondary', 'use_now'),
        'sellinggeneralandadministrativeexpense': ('income_statement', 'expenses', 'selling_general_admin', 'secondary', 'use_now'),
        'operatingexpenses': ('income_statement', 'expenses', 'operating_expenses', 'secondary', 'use_now'),
        'operatingincomeloss': ('income_statement', 'profitability', 'operating_income', 'core', 'use_now'),
        'netincomeloss': ('income_statement', 'profitability', 'net_income', 'core', 'use_now'),
        'earningspersharebasic': ('per_share', 'eps', 'eps_basic', 'core', 'use_now'),
        'earningspersharediluted': ('per_share', 'eps', 'eps_diluted', 'core', 'use_now'),
        'assets': ('balance_sheet', 'assets', 'total_assets', 'core', 'use_now'),
        'assetscurrent': ('balance_sheet', 'assets', 'current_assets', 'core', 'use_now'),
        'liabilities': ('balance_sheet', 'liabilities', 'total_liabilities', 'core', 'use_now'),
        'liabilitiescurrent': ('balance_sheet', 'liabilities', 'current_liabilities', 'core', 'use_now'),
        'stockholdersequity': ('balance_sheet', 'equity', 'stockholders_equity', 'core', 'use_now'),
        'stockholdersequityincludingportionattributabletononcontrollinginterest': ('balance_sheet', 'equity', 'stockholders_equity_incl_nci', 'secondary', 'use_now'),
        'cashandcashequivalentsatcarryingvalue': ('balance_sheet', 'liquidity', 'cash_and_equivalents', 'core', 'use_now'),
        'longtermdebtnoncurrent': ('balance_sheet', 'debt', 'long_term_debt', 'core', 'use_now'),
        'longtermdebtandcapitalleaseobligations': ('balance_sheet', 'debt', 'long_term_debt_and_capital_leases', 'secondary', 'use_now'),
        'shorttermborrowings': ('balance_sheet', 'debt', 'short_term_borrowings', 'secondary', 'use_now'),
        'commonstocksharesoutstanding': ('share_metrics', 'share_count', 'shares_outstanding', 'core', 'use_now'),
        'weightedaveragenumberofsharesoutstandingbasic': ('share_metrics', 'share_count', 'weighted_avg_shares_basic', 'core', 'use_now'),
        'weightedaveragenumberofdilutedsharesoutstanding': ('share_metrics', 'share_count', 'weighted_avg_shares_diluted', 'core', 'use_now'),
        'netcashprovidedbyusedinoperatingactivities': ('cash_flow', 'cash_generation', 'operating_cash_flow', 'core', 'use_now'),
        'paymentstoacquirepropertyplantandequipment': ('cash_flow', 'capital_spending', 'capital_expenditures', 'core', 'use_now'),
        'depreciationdepletionandamortization': ('cash_flow', 'non_cash_items', 'depreciation_amortization', 'secondary', 'use_now'),
    }
    if lower in exact_map:
        statement_group, fact_group, canonical_metric, priority_tier, review_action = exact_map[lower]
        return {
            'statement_group': statement_group,
            'fact_group': fact_group,
            'canonical_metric': canonical_metric,
            'priority_tier': priority_tier,
            'review_action': review_action,
            'tag_reason': 'exact_match',
        }

    pattern_rules = [
        (r'revenue|sales', ('income_statement', 'revenue', 'revenue_related', 'core', 'use_now', 'pattern_revenue')),
        (r'grossprofit|operatingincome|netincome|profit', ('income_statement', 'profitability', 'profitability_related', 'core', 'use_now', 'pattern_profitability')),
        (r'expense|costof|marketing|advertising|fulfillment|restructuring', ('income_statement', 'expenses', 'expense_related', 'secondary', 'use_now', 'pattern_expense')),
        (r'cash|cashequivalent|marketablesecurit', ('balance_sheet', 'liquidity', 'cash_or_investments', 'core', 'use_now', 'pattern_cash')),
        (r'asset', ('balance_sheet', 'assets', 'asset_related', 'secondary', 'use_now', 'pattern_assets')),
        (r'liabilit|payable|accrued', ('balance_sheet', 'liabilities', 'liability_related', 'secondary', 'use_now', 'pattern_liabilities')),
        (r'debt|borrowing|notespayable|creditfacility', ('balance_sheet', 'debt', 'debt_related', 'core', 'use_now', 'pattern_debt')),
        (r'equity|stockholder|retainedearnings|additionalpaidincapital|accumulatedothercomprehensive', ('balance_sheet', 'equity', 'equity_related', 'secondary', 'use_now', 'pattern_equity')),
        (r'shares|stock|eps|earningspershare|sharebasedcompensation', ('share_metrics', 'share_metrics', 'share_related', 'secondary', 'use_now', 'pattern_shares')),
        (r'dividend|repurchase', ('capital_allocation', 'capital_returns', 'capital_return_related', 'secondary', 'use_now', 'pattern_capital_returns')),
        (r'netcash|cashprovided|cashused|capitalexpenditure|propertyplantandequipment', ('cash_flow', 'cash_flow', 'cash_flow_related', 'core', 'use_now', 'pattern_cash_flow')),
        (r'tax|deferredtax', ('tax', 'taxes', 'tax_related', 'secondary', 'use_later', 'pattern_tax')),
        (r'lease|rightofuse', ('operating_details', 'leases', 'lease_related', 'long_tail', 'use_later', 'pattern_lease')),
        (r'fairvalue|derivative|hedg', ('risk_and_exposure', 'market_exposure', 'fair_value_or_derivative', 'long_tail', 'use_later', 'pattern_exposure')),
        (r'goodwill|intangible', ('balance_sheet', 'intangibles', 'intangibles_related', 'secondary', 'use_now', 'pattern_intangibles')),
        (r'inventory|receivable|payable', ('working_capital', 'working_capital', 'working_capital_related', 'secondary', 'use_now', 'pattern_working_capital')),
        (r'segment|geograph|customer|supplier', ('operating_details', 'business_mix', 'segment_or_customer_related', 'long_tail', 'use_later', 'pattern_business_mix')),
    ]

    for pattern, values in pattern_rules:
        if re.search(pattern, lower):
            statement_group, fact_group, canonical_metric, priority_tier, review_action, tag_reason = values
            return {
                'statement_group': statement_group,
                'fact_group': fact_group,
                'canonical_metric': canonical_metric,
                'priority_tier': priority_tier,
                'review_action': review_action,
                'tag_reason': tag_reason,
            }

    return {
        'statement_group': 'other',
        'fact_group': 'long_tail',
        'canonical_metric': 'unmapped_long_tail',
        'priority_tier': 'long_tail',
        'review_action': 'review_later',
        'tag_reason': 'fallback_unmapped',
    }

gdelt_articles = pd.DataFrame()
transactions = pd.read_csv(RAW_DIR / 'fake_mantis_invest.csv')
ticker_universe = sorted({
    str(value).strip()
    for value in transactions.get('Instrument', pd.Series(dtype='object')).dropna().tolist()
    if str(value).strip()
})
len(ticker_universe), ticker_universe[:10]

(12,
 ['AAPL',
  'AMZN',
  'AVGO',
  'BRK.B',
  'GOOGL',
  'META',
  'MSFT',
  'NVDA',
  'QQQ',
  'TSLA'])

## 1. Full Ticker Universe

Build the full stock / ETF universe from the fake Mantis dataset so every later source review is grounded in the actual portfolio universe.


In [16]:
ticker_overview = pd.DataFrame({'ticker': ticker_universe})
ticker_overview['is_blank'] = ticker_overview['ticker'].eq('')
ticker_overview.head(20)


,ticker,is_blank
0,AAPL,False
1,AMZN,False
2,AVGO,False
3,BRK.B,False
4,GOOGL,False
5,META,False
6,MSFT,False
7,NVDA,False
8,QQQ,False
9,TSLA,False


## 2. SEC Company Master + Filing Metadata For Full Universe

This creates the official SEC mapping from ticker to CIK, then collects the recent filing metadata for every mapped company.


In [17]:
company_tickers_url = 'https://www.sec.gov/files/company_tickers.json'
company_tickers = fetch_json(company_tickers_url, headers=SEC_HEADERS)
company_master = pd.DataFrame(company_tickers).T.rename(columns={'ticker': 'ticker', 'title': 'company_name', 'cik_str': 'cik'})
company_master['ticker'] = company_master['ticker'].astype(str).str.upper()
company_master['cik'] = company_master['cik'].astype(str).str.zfill(10)
company_master = company_master[company_master['ticker'].isin(ticker_universe)].copy()
company_master.to_csv(EXTERNAL_DIR / 'sec' / 'company_master_full_universe.csv', index=False)
company_master.head(20)


,cik,ticker,company_name
0,0001045810,NVDA,NVIDIA CORP
1,0000320193,AAPL,Apple Inc.
2,0001652044,GOOGL,Alphabet Inc.
3,0000789019,MSFT,MICROSOFT CORP
4,0001018724,AMZN,AMAZON COM INC
5,0001730168,AVGO,Broadcom Inc.
6,0001326801,META,"Meta Platforms, Inc."
7,0001318605,TSLA,"Tesla, Inc."
14,0001403161,V,VISA INC.
51,0001067839,QQQ,"INVESCO QQQ TRUST, SERIES 1"


In [18]:
filing_frames = []
for row in company_master.itertuples(index=False):
    submissions_url = f'https://data.sec.gov/submissions/CIK{row.cik}.json'
    submissions = fetch_json(submissions_url, headers=SEC_HEADERS)
    recent = pd.DataFrame(submissions.get('filings', {}).get('recent', {}))
    if recent.empty:
        continue
    recent = recent[['accessionNumber', 'filingDate', 'form', 'primaryDocument']].copy()
    recent['ticker'] = row.ticker
    recent['company_name'] = row.company_name
    recent['cik'] = row.cik
    recent['primary_document_url'] = recent.apply(
        lambda r: build_primary_doc_url(row.cik, r['accessionNumber'], r['primaryDocument']),
        axis=1,
    )
    filing_frames.append(recent)

filings_full = pd.concat(filing_frames, ignore_index=True) if filing_frames else pd.DataFrame()
filings_full.to_csv(EXTERNAL_DIR / 'sec' / 'filings_full_universe.csv', index=False)
filings_full.head(20)


,accessionNumber,filingDate,form,primaryDocument,ticker,company_name,cik,primary_document_url
0,0000102909-26-000426,2026-03-26,SCHEDULE 13G/A,xslSCHEDULE_13G_X02/primary_doc.xml,NVDA,NVIDIA CORP,0001045810,https://www.sec.gov/Archives/edgar/data/104581...
1,0001199039-26-000003,2026-03-24,4,xslF345X06/wk-form4_1774386816.xml,NVDA,NVIDIA CORP,0001045810,https://www.sec.gov/Archives/edgar/data/104581...
2,0001725292-26-000002,2026-03-20,4,xslF345X06/wk-form4_1774052040.xml,NVDA,NVIDIA CORP,0001045810,https://www.sec.gov/Archives/edgar/data/104581...
3,0001526111-26-000005,2026-03-20,4,xslF345X06/wk-form4_1774051982.xml,NVDA,NVIDIA CORP,0001045810,https://www.sec.gov/Archives/edgar/data/104581...
4,0001696841-26-000006,2026-03-20,4,xslF345X06/wk-form4_1774051862.xml,NVDA,NVIDIA CORP,0001045810,https://www.sec.gov/Archives/edgar/data/104581...
5,0001283854-26-000006,2026-03-20,4,xslF345X06/wk-form4_1774051812.xml,NVDA,NVIDIA CORP,0001045810,https://www.sec.gov/Archives/edgar/data/104581...
6,0001347842-26-000011,2026-03-20,4,xslF345X06/wk-form4_1774051696.xml,NVDA,NVIDIA CORP,0001045810,https://www.sec.gov/Archives/edgar/data/104581...
7,0001588670-26-000010,2026-03-20,4,xslF345X06/wk-form4_1774051632.xml,NVDA,NVIDIA CORP,0001045810,https://www.sec.gov/Archives/edgar/data/104581...
8,0001197649-26-000005,2026-03-20,4,xslF345X06/wk-form4_1774051558.xml,NVDA,NVIDIA CORP,0001045810,https://www.sec.gov/Archives/edgar/data/104581...
9,0001628280-26-020268,2026-03-20,144,xsl144X01/primary_doc.xml,NVDA,NVIDIA CORP,0001045810,https://www.sec.gov/Archives/edgar/data/104581...


## 3. Latest 10-K / 10-Q URLs + SEC Company Facts

This section builds the filing URLs, a company-facts availability summary, the full flattened fact dataset, and a fact-tag dictionary so we can make sense of the long SEC tag list.


In [19]:
latest_reporting_filings = (
    filings_full[filings_full['form'].isin(['10-K', '10-Q'])]
    .sort_values(['ticker', 'filingDate'], ascending=[True, False])
    .drop_duplicates(subset=['ticker'], keep='first')
    .reset_index(drop=True)
)
latest_reporting_filings.to_csv(EXTERNAL_DIR / 'sec' / 'latest_reporting_filings.csv', index=False)
latest_reporting_filings[['ticker', 'form', 'filingDate', 'primaryDocument', 'primary_document_url']].head(20)


,ticker,form,filingDate,primaryDocument,primary_document_url
0,AAPL,10-Q,2026-01-30,aapl-20251227.htm,https://www.sec.gov/Archives/edgar/data/320193...
1,AMZN,10-K,2026-02-06,amzn-20251231.htm,https://www.sec.gov/Archives/edgar/data/101872...
2,AVGO,10-Q,2026-03-11,avgo-20260201.htm,https://www.sec.gov/Archives/edgar/data/173016...
3,GOOGL,10-K,2026-02-05,goog-20251231.htm,https://www.sec.gov/Archives/edgar/data/165204...
4,META,10-K,2026-01-29,meta-20251231.htm,https://www.sec.gov/Archives/edgar/data/132680...
5,MSFT,10-Q,2026-01-28,msft-20251231.htm,https://www.sec.gov/Archives/edgar/data/789019...
6,NVDA,10-K,2026-02-25,nvda-20260125.htm,https://www.sec.gov/Archives/edgar/data/104581...
7,TSLA,10-K,2026-01-29,tsla-20251231.htm,https://www.sec.gov/Archives/edgar/data/131860...
8,V,10-Q,2026-01-30,v-20251231.htm,https://www.sec.gov/Archives/edgar/data/140316...


In [20]:
company_fact_rows = []
company_fact_detail_rows = []
for row in company_master.itertuples(index=False):
    facts_url = f'https://data.sec.gov/api/xbrl/companyfacts/CIK{row.cik}.json'
    try:
        company_facts = fetch_json(facts_url, headers=SEC_HEADERS)
        us_gaap = company_facts.get('facts', {}).get('us-gaap', {})
        company_fact_rows.append({
            'ticker': row.ticker,
            'company_name': row.company_name,
            'cik': row.cik,
            'facts_status': 'available',
            'fact_tag_count': len(us_gaap),
            'sample_fact_tags': ', '.join(list(us_gaap.keys())[:10]),
        })

        for fact_tag, fact_payload in us_gaap.items():
            units = fact_payload.get('units', {})
            for unit_name, observations in units.items():
                for observation in observations:
                    company_fact_detail_rows.append({
                        'ticker': row.ticker,
                        'company_name': row.company_name,
                        'cik': row.cik,
                        'fact_tag': fact_tag,
                        'unit': unit_name,
                        'value': observation.get('val'),
                        'filed_date': observation.get('filed'),
                        'fiscal_year': observation.get('fy'),
                        'fiscal_period': observation.get('fp'),
                        'frame': observation.get('frame'),
                        'form': observation.get('form'),
                        'fact_start': observation.get('start'),
                        'fact_end': observation.get('end'),
                    })
    except HTTPError as error:
        company_fact_rows.append({
            'ticker': row.ticker,
            'company_name': row.company_name,
            'cik': row.cik,
            'facts_status': f'http_{error.code}',
            'fact_tag_count': 0,
            'sample_fact_tags': '',
        })

company_facts_review = pd.DataFrame(company_fact_rows)
company_facts_review.to_csv(EXTERNAL_DIR / 'sec' / 'company_facts_review.csv', index=False)

company_facts_full_df = pd.DataFrame(company_fact_detail_rows)
if not company_facts_full_df.empty:
    fact_tag_meta = company_facts_full_df['fact_tag'].apply(classify_fact_tag).apply(pd.Series)
    company_facts_full_df = pd.concat([company_facts_full_df, fact_tag_meta], axis=1)
company_facts_full_df.to_csv(EXTERNAL_DIR / 'sec' / 'company_facts_full.csv', index=False)

if not company_facts_full_df.empty:
    fact_tag_dictionary_df = (
        company_facts_full_df.groupby('fact_tag', dropna=False)
        .agg(
            statement_group=('statement_group', 'first'),
            fact_group=('fact_group', 'first'),
            canonical_metric=('canonical_metric', 'first'),
            priority_tier=('priority_tier', 'first'),
            review_action=('review_action', 'first'),
            tag_reason=('tag_reason', 'first'),
            row_count=('fact_tag', 'size'),
            company_count=('ticker', 'nunique'),
            unit_count=('unit', 'nunique'),
            sample_units=('unit', lambda s: ', '.join(sorted({str(v) for v in s.dropna().astype(str).tolist()[:5]}))),
            sample_tickers=('ticker', lambda s: ', '.join(sorted({str(v) for v in s.dropna().astype(str).tolist()[:5]}))),
        )
        .reset_index()
        .sort_values(['priority_tier', 'company_count', 'row_count', 'fact_tag'], ascending=[True, False, False, True])
        .reset_index(drop=True)
    )
else:
    fact_tag_dictionary_df = pd.DataFrame()
fact_tag_dictionary_df.to_csv(EXTERNAL_DIR / 'sec' / 'fact_tag_dictionary.csv', index=False)

display(company_facts_review.head(20))
display(company_facts_full_df.head(20) if not company_facts_full_df.empty else pd.DataFrame())
fact_tag_dictionary_df.head(50) if not fact_tag_dictionary_df.empty else pd.DataFrame()


,ticker,company_name,cik,facts_status,fact_tag_count,sample_fact_tags
0,NVDA,NVIDIA CORP,0001045810,available,625,"AcceleratedShareRepurchaseProgramAdjustment, A..."
1,AAPL,Apple Inc.,0000320193,available,503,"AccountsPayable, AccountsPayableCurrent, Accou..."
2,GOOGL,Alphabet Inc.,0001652044,available,521,"AccountsPayableCurrent, AccountsReceivableNetC..."
3,MSFT,MICROSOFT CORP,0000789019,available,543,"AccountsPayableCurrent, AccountsReceivableNet,..."
4,AMZN,AMAZON COM INC,0001018724,available,538,"AccountsPayable, AccountsPayableCurrent, Accou..."
5,AVGO,Broadcom Inc.,0001730168,available,432,"AccountsPayableCurrent, AccountsReceivableNetC..."
6,META,"Meta Platforms, Inc.",0001326801,available,455,AccountsPayableAndOtherAccruedLiabilitiesCurre...
7,TSLA,"Tesla, Inc.",0001318605,available,638,"AccountsAndNotesReceivableNet, AccountsPayable..."
8,V,VISA INC.,0001403161,available,625,"AccountsPayable, AccountsPayableCurrent, Accou..."
9,QQQ,"INVESCO QQQ TRUST, SERIES 1",0001067839,http_404,0,


,ticker,company_name,cik,fact_tag,unit,value,filed_date,fiscal_year,fiscal_period,frame,form,fact_start,fact_end,statement_group,fact_group,canonical_metric,priority_tier,review_action,tag_reason
0,NVDA,NVIDIA CORP,0001045810,AcceleratedShareRepurchaseProgramAdjustment,USD,9.000000e+06,2016-08-23,2017.0,Q2,CY2016Q2,10-Q,2016-05-02,2016-07-31,capital_allocation,capital_returns,capital_return_related,secondary,use_now,pattern_capital_returns
1,NVDA,NVIDIA CORP,0001045810,AcceleratedShareRepurchaseProgramAdjustment,USD,9.000000e+06,2016-11-22,2017.0,Q3,None,10-Q,2016-02-01,2016-10-30,capital_allocation,capital_returns,capital_return_related,secondary,use_now,pattern_capital_returns
2,NVDA,NVIDIA CORP,0001045810,AcceleratedShareRepurchasesFinalPricePaidPerShare,USD/shares,2.163000e+01,2015-11-18,2016.0,Q3,None,10-Q,2015-01-26,2015-10-25,capital_allocation,capital_returns,capital_return_related,secondary,use_now,pattern_capital_returns
3,NVDA,NVIDIA CORP,0001045810,AcceleratedShareRepurchasesFinalPricePaidPerShare,USD/shares,4.206000e+01,2016-08-23,2017.0,Q2,None,10-Q,2016-02-01,2016-07-31,capital_allocation,capital_returns,capital_return_related,secondary,use_now,pattern_capital_returns
4,NVDA,NVIDIA CORP,0001045810,AcceleratedShareRepurchasesFinalPricePaidPerShare,USD/shares,4.206000e+01,2016-11-22,2017.0,Q3,None,10-Q,2016-02-01,2016-10-30,capital_allocation,capital_returns,capital_return_related,secondary,use_now,pattern_capital_returns
5,NVDA,NVIDIA CORP,0001045810,AcceleratedShareRepurchasesSettlementPaymentOr...,USD,7.500000e+08,2013-05-22,2013.0,Q1,None,10-Q,None,2013-07-28,capital_allocation,capital_returns,capital_return_related,secondary,use_now,pattern_capital_returns
6,NVDA,NVIDIA CORP,0001045810,AcceleratedShareRepurchasesSettlementPaymentOr...,USD,7.500000e+08,2013-08-21,2013.0,Q2,CY2013Q2I,10-Q,None,2013-07-28,capital_allocation,capital_returns,capital_return_related,secondary,use_now,pattern_capital_returns
7,NVDA,NVIDIA CORP,0001045810,AcceleratedShareRepurchasesSettlementPaymentOr...,USD,7.500000e+08,2013-11-19,2013.0,Q3,CY2013Q3I,10-Q,None,2013-10-27,capital_allocation,capital_returns,capital_return_related,secondary,use_now,pattern_capital_returns
8,NVDA,NVIDIA CORP,0001045810,AcceleratedShareRepurchasesSettlementPaymentOr...,USD,4.000000e+08,2015-11-18,2016.0,Q3,CY2015Q3I,10-Q,None,2015-10-25,capital_allocation,capital_returns,capital_return_related,secondary,use_now,pattern_capital_returns
9,NVDA,NVIDIA CORP,0001045810,AcceleratedShareRepurchasesSettlementPaymentOr...,USD,5.870000e+08,2016-03-17,2016.0,FY,CY2015Q4I,10-K,None,2016-01-31,capital_allocation,capital_returns,capital_return_related,secondary,use_now,pattern_capital_returns


,fact_tag,statement_group,fact_group,canonical_metric,priority_tier,review_action,tag_reason,row_count,company_count,unit_count,sample_units,sample_tickers
0,NetIncomeLoss,income_statement,profitability,net_income,core,use_now,exact_match,2236,9,1,USD,NVDA
1,OperatingIncomeLoss,income_statement,profitability,operating_income,core,use_now,exact_match,1879,9,1,USD,NVDA
2,CashAndCashEquivalentsAtCarryingValue,balance_sheet,liquidity,cash_and_equivalents,core,use_now,exact_match,1828,9,1,USD,NVDA
3,StockholdersEquity,balance_sheet,equity,stockholders_equity,core,use_now,exact_match,1491,9,1,USD,NVDA
4,OtherNonoperatingIncomeExpense,income_statement,profitability,profitability_related,core,use_now,pattern_profitability,1454,9,1,USD,NVDA
5,NetCashProvidedByUsedInFinancingActivities,balance_sheet,liquidity,cash_or_investments,core,use_now,pattern_cash,1320,9,1,USD,NVDA
6,NetCashProvidedByUsedInOperatingActivities,cash_flow,cash_generation,operating_cash_flow,core,use_now,exact_match,1320,9,1,USD,NVDA
7,NetCashProvidedByUsedInInvestingActivities,balance_sheet,liquidity,cash_or_investments,core,use_now,pattern_cash,1311,9,1,USD,NVDA
8,Assets,balance_sheet,assets,total_assets,core,use_now,exact_match,1078,9,1,USD,NVDA
9,AssetsCurrent,balance_sheet,assets,current_assets,core,use_now,exact_match,1062,9,1,USD,NVDA


## 4. Inflate SEC Filing URLs Into Full Text + Review

This section now keeps both review tables and a fuller extracted text dataset for the inflated SEC filing URLs.


In [21]:
sec_document_inventory_rows = []
sec_document_text_rows = []
sec_cleaning_review_rows = []
sec_section_preview_rows = []

for row in latest_reporting_filings.head(MAX_SEC_DOCS_TO_INFLATE).itertuples(index=False):
    html = fetch_text(row.primary_document_url, headers=SEC_HEADERS)
    raw_text = html_to_text(html)
    clean_text = clean_filing_text(raw_text)
    raw_chunks = simple_chunk(raw_text)
    clean_chunks = simple_chunk(clean_text)
    section_previews = get_filing_section_previews(clean_text, row.form)
    selected_text = section_previews['risk_factors_preview'] or section_previews['mda_preview'] or section_previews['business_preview'] or (clean_chunks[0] if clean_chunks else '')

    sec_document_inventory_rows.append({
        'ticker': row.ticker,
        'form': row.form,
        'filing_date': row.filingDate,
        'primary_document_url': row.primary_document_url,
        'fetch_status': 'fetched',
        'raw_document_char_count': len(raw_text),
    })

    sec_document_text_rows.append({
        'ticker': row.ticker,
        'form': row.form,
        'filing_date': row.filingDate,
        'primary_document_url': row.primary_document_url,
        'raw_text': raw_text,
        'clean_text': clean_text,
        'risk_factors_text': section_previews['risk_factors_preview'],
        'mda_text': section_previews['mda_preview'],
        'business_text': section_previews['business_preview'],
        'selected_clean_text': selected_text,
    })

    sec_cleaning_review_rows.append({
        'ticker': row.ticker,
        'form': row.form,
        'filing_date': row.filingDate,
        'raw_document_char_count': len(raw_text),
        'clean_document_char_count': len(clean_text),
        'raw_chunk_count': len(raw_chunks),
        'clean_chunk_count': len(clean_chunks),
        'contains_risk_word': 'risk' in clean_text.lower(),
        'contains_management_word': 'management' in clean_text.lower(),
        'raw_text_preview': (raw_chunks[0][:800] if raw_chunks else ''),
        'clean_text_preview': (clean_chunks[0][:1200] if clean_chunks else ''),
    })

    sec_section_preview_rows.append({
        'ticker': row.ticker,
        'form': row.form,
        'filing_date': row.filingDate,
        'risk_factors_preview': section_previews['risk_factors_preview'][:1200],
        'mda_preview': section_previews['mda_preview'][:1200],
        'business_preview': section_previews['business_preview'][:1200],
        'selected_clean_preview': selected_text[:1200],
    })

sec_document_inventory_df = pd.DataFrame(sec_document_inventory_rows)
sec_document_inventory_df.to_csv(REVIEW_DIR / 'sec_document_inventory_review.csv', index=False)

sec_document_text_full_df = pd.DataFrame(sec_document_text_rows)
sec_document_text_full_df.to_csv(REVIEW_DIR / 'sec_document_text_full.csv', index=False)

sec_cleaning_review_df = pd.DataFrame(sec_cleaning_review_rows)
sec_cleaning_review_df.to_csv(REVIEW_DIR / 'sec_cleaning_review.csv', index=False)

sec_section_preview_df = pd.DataFrame(sec_section_preview_rows)
sec_section_preview_df.to_csv(REVIEW_DIR / 'sec_section_preview_review.csv', index=False)

display(sec_document_inventory_df.head(10))
display(sec_cleaning_review_df[['ticker', 'form', 'raw_document_char_count', 'clean_document_char_count', 'clean_chunk_count', 'clean_text_preview']].head(10))
display(sec_section_preview_df[['ticker', 'form', 'risk_factors_preview', 'mda_preview', 'business_preview', 'selected_clean_preview']].head(10))
sec_document_text_full_df[['ticker', 'form', 'filing_date', 'selected_clean_text']].head(10)


,ticker,form,filing_date,primary_document_url,fetch_status,raw_document_char_count
0,AAPL,10-Q,2026-01-30,https://www.sec.gov/Archives/edgar/data/320193...,fetched,72361
1,AMZN,10-K,2026-02-06,https://www.sec.gov/Archives/edgar/data/101872...,fetched,326605
2,AVGO,10-Q,2026-03-11,https://www.sec.gov/Archives/edgar/data/173016...,fetched,203589
3,GOOGL,10-K,2026-02-05,https://www.sec.gov/Archives/edgar/data/165204...,fetched,398500
4,META,10-K,2026-01-29,https://www.sec.gov/Archives/edgar/data/132680...,fetched,549559
5,MSFT,10-Q,2026-01-28,https://www.sec.gov/Archives/edgar/data/789019...,fetched,312016
6,NVDA,10-K,2026-02-25,https://www.sec.gov/Archives/edgar/data/104581...,fetched,373320
7,TSLA,10-K,2026-01-29,https://www.sec.gov/Archives/edgar/data/131860...,fetched,444095
8,V,10-Q,2026-01-30,https://www.sec.gov/Archives/edgar/data/140316...,fetched,109473


,ticker,form,raw_document_char_count,clean_document_char_count,clean_chunk_count,clean_text_preview
0,AAPL,10-Q,72361,67605,38,aapl-20251227 false 2026 Q1 0000320193 --09-26...
1,AMZN,10-K,326605,312999,174,amzn-20251231 false 2025 FY 0001018724 P4Y0M P...
2,AVGO,10-Q,203589,199681,111,avgo-20260201 0001730168 November 1 2026 Q1 FA...
3,GOOGL,10-K,398500,376266,210,goog-20251231 FALSE 2025 FY 0001652044 P7Y P2Y...
4,META,10-K,549559,537874,299,meta-20251231 false 2025 FY 0001326801 P5Y P1Y...
5,MSFT,10-Q,312016,298447,166,10-Q --06-30 0000789019 Q2 false 2026 2014 201...
6,NVDA,10-K,373320,364053,203,nvda-20260125 0001045810 2026 FY false 362 460...
7,TSLA,10-K,444095,430821,240,tsla-20251231 false 0001318605 2025 FY P3Y P3Y...
8,V,10-Q,109473,100446,56,v-20251231 0001403161 9/30 2026 Q1 false 50 50...


,ticker,form,risk_factors_preview,mda_preview,business_preview,selected_clean_preview
0,AAPL,10-Q,Item 1A. Risk Factors 20,,business exposure to foreign exchange and inte...,Item 1A. Risk Factors 20
1,AMZN,10-K,Item 1A. Risk Factors 6,,,Item 1A. Risk Factors 6
2,AVGO,10-Q,Risk Factors 29 Item&#160;2. Unregistered Sale...,,"business, including but not limited to commerc...",Risk Factors 29 Item&#160;2. Unregistered Sale...
3,GOOGL,10-K,Risk Factors 9 Item&#160;1B. Unresolved Staff ...,Management's Discussion and Analysis of Financ...,ITEM 1. BUSINESS Overview As our founders Larr...,Risk Factors 9 Item&#160;1B. Unresolved Staff ...
4,META,10-K,Item 1A. Risk Factors 12,Management's Discussion and Analysis of Financ...,Item 1. Business 6 Item 1A. Risk Factors 12,Item 1A. Risk Factors 12
5,MSFT,10-Q,Item 1A. Risk Factors 47 &#160; &#160; &#160; ...,,BusinessProcessesMember 2024-07-01 2024-12-31 ...,Item 1A. Risk Factors 47 &#160; &#160; &#160; ...
6,NVDA,10-K,Item 1A. Risk Factors &#160; 12,Item 7. Management's Discussion and Analysis o...,Item 1. Business &#160; 4,Item 1A. Risk Factors &#160; 12
7,TSLA,10-K,Item 1A. Risk Factors 12,Item 7. Management's Discussion and Analysis o...,Item 1. Business 2 Item 1A. Risk Factors 12,Item 1A. Risk Factors 12
8,V,10-Q,Risk Factors 32 Item&#160;2. Unregistered Sale...,,business. The Company has one reportable segme...,Risk Factors 32 Item&#160;2. Unregistered Sale...


,ticker,form,filing_date,selected_clean_text
0,AAPL,10-Q,2026-01-30,Item 1A. Risk Factors 20
1,AMZN,10-K,2026-02-06,Item 1A. Risk Factors 6
2,AVGO,10-Q,2026-03-11,Risk Factors 29 Item&#160;2. Unregistered Sale...
3,GOOGL,10-K,2026-02-05,Risk Factors 9 Item&#160;1B. Unresolved Staff ...
4,META,10-K,2026-01-29,Item 1A. Risk Factors 12
5,MSFT,10-Q,2026-01-28,Item 1A. Risk Factors 47 &#160; &#160; &#160; ...
6,NVDA,10-K,2026-02-25,Item 1A. Risk Factors &#160; 12
7,TSLA,10-K,2026-01-29,Item 1A. Risk Factors 12
8,V,10-Q,2026-01-30,Risk Factors 32 Item&#160;2. Unregistered Sale...


## 5. FRED Macro Layer For Economic Weather

Fetch the macro series that help describe the broader economic backdrop.


In [22]:
FRED_API_KEY = os.getenv('FRED_API_KEY', '')
fred_series = {
    'FEDFUNDS': 'Fed Funds Rate',
    'CPIAUCSL': 'CPI',
    'UNRATE': 'Unemployment Rate',
    'DGS10': '10Y Treasury',
    'DGS2': '2Y Treasury',
}

fred_frames = []
for series_id, title in fred_series.items():
    fred_url = (
        'https://api.stlouisfed.org/fred/series/observations'
        f'?series_id={series_id}&api_key={FRED_API_KEY}&file_type=json'
    )
    payload = fetch_json(fred_url)
    frame = pd.DataFrame(payload.get('observations', []))[['date', 'value']]
    frame['series_id'] = series_id
    frame['title'] = title
    fred_frames.append(frame)

fred_df = pd.concat(fred_frames, ignore_index=True) if fred_frames else pd.DataFrame()
fred_df.to_csv(EXTERNAL_DIR / 'fred' / 'macro_series_full_review.csv', index=False)
fred_df.tail(20)


,date,value,series_id,title
32507,2026-03-13,3.73,DGS2,2Y Treasury
32508,2026-03-16,3.68,DGS2,2Y Treasury
32509,2026-03-17,3.68,DGS2,2Y Treasury
32510,2026-03-18,3.76,DGS2,2Y Treasury
32511,2026-03-19,3.79,DGS2,2Y Treasury
32512,2026-03-20,3.88,DGS2,2Y Treasury
32513,2026-03-23,3.83,DGS2,2Y Treasury
32514,2026-03-24,3.9,DGS2,2Y Treasury
32515,2026-03-25,3.84,DGS2,2Y Treasury
32516,2026-03-26,3.96,DGS2,2Y Treasury


## 6. FMP Company Profile Enrichment For Full Universe

Use the current stable FMP profile endpoint to enrich the full ticker universe with sector, industry, market cap, and instrument type hints.


In [23]:
FMP_API_KEY = os.getenv('FMP_API_KEY', '')
profile_frames = []
for symbol in ticker_universe:
    fmp_url = f'https://financialmodelingprep.com/stable/profile?symbol={symbol}&apikey={FMP_API_KEY}'
    profile_payload = fetch_json(fmp_url)
    profile_frames.append(pd.DataFrame(profile_payload if isinstance(profile_payload, list) else [profile_payload]))

fmp_profiles = pd.concat(profile_frames, ignore_index=True) if profile_frames else pd.DataFrame()
fmp_profiles.to_csv(EXTERNAL_DIR / 'fmp' / 'company_profiles_full_universe.csv', index=False)
display_columns = [col for col in ['symbol', 'companyName', 'sector', 'industry', 'marketCap', 'isEtf', 'isFund'] if col in fmp_profiles.columns]
fmp_profiles[display_columns].head(20)


,symbol,companyName,sector,industry,marketCap,isEtf,isFund
0,AAPL,Apple Inc.,Technology,Consumer Electronics,3828515423772,False,False
1,AMZN,"Amazon.com, Inc.",Consumer Cyclical,Specialty Retail,2558990436991,False,False
2,AVGO,Broadcom Inc.,Technology,Semiconductors,1761620156664,False,False
3,GOOGL,Alphabet Inc.,Communication Services,Internet Content & Information,3837652369144,False,False
4,META,"Meta Platforms, Inc.",Communication Services,Internet Content & Information,1587872631454,False,False
5,MSFT,Microsoft Corporation,Technology,Software - Infrastructure,2753943398100,False,False
6,NVDA,NVIDIA Corporation,Technology,Semiconductors,4584652476141,False,False
7,QQQ,"Invesco QQQ Trust, Series 1",Financial Services,Asset Management,375747842495,True,False
8,TSLA,"Tesla, Inc.",Consumer Cyclical,Auto - Manufacturers,1309411140817,False,False
9,V,Visa Inc.,Financial Services,Financial - Credit Services,586818164309,False,False


## 7. GDELT News Metadata For Full Universe

This section uses on-disk caching, graceful rate-limit handling, and response-side filtering so we keep English-language, U.S.-source articles without relying on a brittle query string. It keeps `time.sleep(5.2)` in place between fresh requests.


In [24]:
news_frames = []
gdelt_fetch_log_rows = []
for symbol in ticker_universe:
    cache_path = EXTERNAL_DIR / 'gdelt' / 'cache' / f'{symbol}_artlist.json'
    payload = load_cached_json(cache_path)
    gdelt_query = symbol
    source_status = 'cache_hit' if payload is not None else 'fresh_request'
    if payload is None:
        gdelt_url = (
            'https://api.gdeltproject.org/api/v2/doc/doc'
            f'?query={quote(gdelt_query)}&mode=artlist&maxrecords={MAX_GDELT_ARTICLES_PER_TICKER}&format=json'
        )
        try:
            payload = fetch_json(gdelt_url)
            write_cached_json(cache_path, payload)
            time.sleep(5.2)
        except HTTPError as error:
            payload = {'articles': []}
            source_status = f'http_{error.code}'
        except Exception as error:
            payload = {'articles': []}
            source_status = f'error_{type(error).__name__}'

    articles = pd.DataFrame(payload.get('articles', []))
    raw_article_count = 0 if articles.empty else len(articles)
    if not articles.empty:
        if 'language' in articles.columns:
            articles = articles[articles['language'].astype(str).str.lower() == GDELT_SOURCE_LANGUAGE.lower()]
        if 'sourcecountry' in articles.columns:
            articles = articles[articles['sourcecountry'].astype(str).str.lower().isin(['united states', GDELT_SOURCE_COUNTRY.lower()])]
    article_count = 0 if articles.empty else len(articles)
    gdelt_fetch_log_rows.append({
        'ticker': symbol,
        'gdelt_query': gdelt_query,
        'fetch_status': source_status,
        'raw_article_count': raw_article_count,
        'article_count': article_count,
        'cache_path': str(cache_path),
    })
    if articles.empty:
        continue
    articles['ticker'] = symbol
    news_frames.append(articles)

gdelt_articles = pd.concat(news_frames, ignore_index=True) if news_frames else pd.DataFrame()
gdelt_articles.to_csv(EXTERNAL_DIR / 'gdelt' / 'articles_full_universe.csv', index=False)

gdelt_fetch_log_df = pd.DataFrame(gdelt_fetch_log_rows)
gdelt_fetch_log_df.to_csv(REVIEW_DIR / 'gdelt_fetch_log.csv', index=False)

display(gdelt_fetch_log_df.head(20))
gdelt_articles[['ticker', 'title', 'domain', 'seendate', 'url']].head(20) if not gdelt_articles.empty else pd.DataFrame()


Rate limited for https://api.gdeltproject.org/api/v2/doc/doc?query=V&mode=artlist&maxrecords=3&format=json. Sleeping 1.5s before retry 2/4.


,ticker,gdelt_query,fetch_status,raw_article_count,article_count,cache_path
0,AAPL,AAPL,cache_hit,3,1,/Users/sagnikrana/Documents/GitHub/portfolio-a...
1,AMZN,AMZN,cache_hit,3,3,/Users/sagnikrana/Documents/GitHub/portfolio-a...
2,AVGO,AVGO,cache_hit,3,3,/Users/sagnikrana/Documents/GitHub/portfolio-a...
3,BRK.B,BRK.B,error_JSONDecodeError,0,0,/Users/sagnikrana/Documents/GitHub/portfolio-a...
4,GOOGL,GOOGL,cache_hit,3,3,/Users/sagnikrana/Documents/GitHub/portfolio-a...
5,META,META,cache_hit,3,3,/Users/sagnikrana/Documents/GitHub/portfolio-a...
6,MSFT,MSFT,cache_hit,3,3,/Users/sagnikrana/Documents/GitHub/portfolio-a...
7,NVDA,NVDA,cache_hit,3,3,/Users/sagnikrana/Documents/GitHub/portfolio-a...
8,QQQ,QQQ,cache_hit,3,3,/Users/sagnikrana/Documents/GitHub/portfolio-a...
9,TSLA,TSLA,cache_hit,3,3,/Users/sagnikrana/Documents/GitHub/portfolio-a...


,ticker,title,domain,seendate,url
0,AAPL,Apple ( AAPL ) Postpones Its Smart Home Displa...,insidermonkey.com,20260315T091500Z,https://www.insidermonkey.com/blog/apple-aapl-...
1,AMZN,Is Amazon . com ( AMZN ) One of the Best AI In...,finance.yahoo.com,20260403T044500Z,https://finance.yahoo.com/markets/stocks/artic...
2,AMZN,Amazon ( AMZN ) Buys George Washington Univers...,insidermonkey.com,20260305T011500Z,https://www.insidermonkey.com/blog/amazon-amzn...
3,AMZN,"BofA Maintains Buy on Amazon . com , Inc . ( A...",insidermonkey.com,20260309T210000Z,https://www.insidermonkey.com/blog/bofa-mainta...
4,AVGO,Analysts Remain Bullish on Broadcom ( AVGO ) W...,finance.yahoo.com,20260124T010000Z,https://finance.yahoo.com/news/analysts-remain...
5,AVGO,Citi Lowers PT on Broadcom Inc . ( AVGO ) Stock,finance.yahoo.com,20260225T084500Z,https://finance.yahoo.com/news/citi-lowers-pt-...
6,AVGO,Anlaysts Bullish on Broadcom ( AVGO ) Despite...,insidermonkey.com,20260401T173000Z,https://www.insidermonkey.com/blog/anlaysts-bu...
7,GOOGL,Investors Are Bullish on Alphabet Stock - Pili...,finance.yahoo.com,20260306T060000Z,https://finance.yahoo.com/news/investors-bulli...
8,GOOGL,Is Alphabet ( GOOGL ) One of the Best Long Ter...,finance.yahoo.com,20260412T213000Z,https://finance.yahoo.com/markets/stocks/artic...
9,GOOGL,Alphabet ( GOOGL ) Sees Optimistic Coverage Fr...,insidermonkey.com,20260317T074500Z,https://www.insidermonkey.com/blog/alphabet-go...


## 8. Inflate News URLs Into Full Text + Review

This section keeps both a lightweight review table and the fuller extracted article text dataset for the inflated news URLs.


In [26]:
gdelt_articles = globals().get('gdelt_articles', pd.DataFrame())
if gdelt_articles.empty:
    cached_articles_path = EXTERNAL_DIR / 'gdelt' / 'articles_full_universe.csv'
    if cached_articles_path.exists():
        gdelt_articles = pd.read_csv(cached_articles_path)

article_reviews = []
article_text_rows = []
if not gdelt_articles.empty:
    unique_articles = (
        gdelt_articles[['ticker', 'title', 'domain', 'seendate', 'url']]
        .dropna(subset=['url'])
        .drop_duplicates(subset=['url'])
        .head(MAX_NEWS_URLS_TO_INFLATE)
    )
    for row in unique_articles.itertuples(index=False):
        try:
            html = fetch_text(row.url)
            raw_text = html_to_text(html)
            clean_text = clean_article_text(raw_text)
            preview = simple_chunk(clean_text, chunk_size=1200)
            selected_text = preview[0] if preview else ''
            article_reviews.append({
                'ticker': row.ticker,
                'domain': row.domain,
                'seendate': row.seendate,
                'url': row.url,
                'title': row.title,
                'fetch_status': 'fetched',
                'raw_char_count': len(raw_text),
                'clean_char_count': len(clean_text),
                'contains_market_words': any(word in clean_text.lower() for word in ['guidance', 'earnings', 'tariff', 'war', 'inflation', 'rate']),
                'text_preview': selected_text[:1200],
            })
            article_text_rows.append({
                'ticker': row.ticker,
                'domain': row.domain,
                'seendate': row.seendate,
                'url': row.url,
                'title': row.title,
                'raw_text': raw_text,
                'clean_text': clean_text,
                'selected_text': selected_text,
            })
        except Exception as error:
            article_reviews.append({
                'ticker': row.ticker,
                'domain': row.domain,
                'seendate': row.seendate,
                'url': row.url,
                'title': row.title,
                'fetch_status': f'error: {type(error).__name__}',
                'raw_char_count': 0,
                'clean_char_count': 0,
                'contains_market_words': False,
                'text_preview': f'ERROR: {error}',
            })
            article_text_rows.append({
                'ticker': row.ticker,
                'domain': row.domain,
                'seendate': row.seendate,
                'url': row.url,
                'title': row.title,
                'raw_text': '',
                'clean_text': '',
                'selected_text': f'ERROR: {error}',
            })

article_review_df = pd.DataFrame(article_reviews)
article_review_df.to_csv(REVIEW_DIR / 'gdelt_article_review.csv', index=False)

article_text_full_df = pd.DataFrame(article_text_rows)
article_text_full_df.to_csv(REVIEW_DIR / 'gdelt_article_text_full.csv', index=False)

display(article_review_df.head(20) if not article_review_df.empty else pd.DataFrame())
article_text_full_df[['ticker', 'domain', 'title', 'selected_text']].head(20) if not article_text_full_df.empty else pd.DataFrame()


Rate limited for https://finance.yahoo.com/markets/stocks/articles/amazon-com-amzn-one-best-164432480.html. Sleeping 1.5s before retry 2/4.
Rate limited for https://finance.yahoo.com/markets/stocks/articles/amazon-com-amzn-one-best-164432480.html. Sleeping 3.0s before retry 3/4.
Rate limited for https://finance.yahoo.com/markets/stocks/articles/amazon-com-amzn-one-best-164432480.html. Sleeping 6.0s before retry 4/4.
Rate limited for https://finance.yahoo.com/news/analysts-remain-bullish-broadcom-avgo-180856909.html. Sleeping 1.5s before retry 2/4.
Rate limited for https://finance.yahoo.com/news/analysts-remain-bullish-broadcom-avgo-180856909.html. Sleeping 3.0s before retry 3/4.
Rate limited for https://finance.yahoo.com/news/analysts-remain-bullish-broadcom-avgo-180856909.html. Sleeping 6.0s before retry 4/4.
Rate limited for https://finance.yahoo.com/news/citi-lowers-pt-broadcom-inc-211530100.html. Sleeping 1.5s before retry 2/4.
Rate limited for https://finance.yahoo.com/news/citi-l

,ticker,domain,seendate,url,title,fetch_status,raw_char_count,clean_char_count,contains_market_words,text_preview
0,AAPL,insidermonkey.com,20260315T091500Z,https://www.insidermonkey.com/blog/apple-aapl-...,Apple ( AAPL ) Postpones Its Smart Home Displa...,fetched,390900,385796,True,"(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({..."
1,AMZN,finance.yahoo.com,20260403T044500Z,https://finance.yahoo.com/markets/stocks/artic...,Is Amazon . com ( AMZN ) One of the Best AI In...,error: HTTPError,0,0,False,ERROR: HTTP Error 429: Too Many Requests
2,AMZN,insidermonkey.com,20260305T011500Z,https://www.insidermonkey.com/blog/amazon-amzn...,Amazon ( AMZN ) Buys George Washington Univers...,fetched,391018,385917,True,"(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({..."
3,AMZN,insidermonkey.com,20260309T210000Z,https://www.insidermonkey.com/blog/bofa-mainta...,"BofA Maintains Buy on Amazon . com , Inc . ( A...",fetched,390840,385747,True,"(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({..."
4,AVGO,finance.yahoo.com,20260124T010000Z,https://finance.yahoo.com/news/analysts-remain...,Analysts Remain Bullish on Broadcom ( AVGO ) W...,error: HTTPError,0,0,False,ERROR: HTTP Error 429: Too Many Requests
5,AVGO,finance.yahoo.com,20260225T084500Z,https://finance.yahoo.com/news/citi-lowers-pt-...,Citi Lowers PT on Broadcom Inc . ( AVGO ) Stock,error: HTTPError,0,0,False,ERROR: HTTP Error 429: Too Many Requests
6,AVGO,insidermonkey.com,20260401T173000Z,https://www.insidermonkey.com/blog/anlaysts-bu...,Anlaysts Bullish on Broadcom ( AVGO ) Despite...,fetched,390928,385840,True,"(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({..."
7,GOOGL,finance.yahoo.com,20260306T060000Z,https://finance.yahoo.com/news/investors-bulli...,Investors Are Bullish on Alphabet Stock - Pili...,error: HTTPError,0,0,False,ERROR: HTTP Error 429: Too Many Requests
8,GOOGL,finance.yahoo.com,20260412T213000Z,https://finance.yahoo.com/markets/stocks/artic...,Is Alphabet ( GOOGL ) One of the Best Long Ter...,error: HTTPError,0,0,False,ERROR: HTTP Error 429: Too Many Requests
9,GOOGL,insidermonkey.com,20260317T074500Z,https://www.insidermonkey.com/blog/alphabet-go...,Alphabet ( GOOGL ) Sees Optimistic Coverage Fr...,fetched,390648,385548,True,"(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({..."


,ticker,domain,title,selected_text
0,AAPL,insidermonkey.com,Apple ( AAPL ) Postpones Its Smart Home Displa...,"(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({..."
1,AMZN,finance.yahoo.com,Is Amazon . com ( AMZN ) One of the Best AI In...,ERROR: HTTP Error 429: Too Many Requests
2,AMZN,insidermonkey.com,Amazon ( AMZN ) Buys George Washington Univers...,"(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({..."
3,AMZN,insidermonkey.com,"BofA Maintains Buy on Amazon . com , Inc . ( A...","(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({..."
4,AVGO,finance.yahoo.com,Analysts Remain Bullish on Broadcom ( AVGO ) W...,ERROR: HTTP Error 429: Too Many Requests
5,AVGO,finance.yahoo.com,Citi Lowers PT on Broadcom Inc . ( AVGO ) Stock,ERROR: HTTP Error 429: Too Many Requests
6,AVGO,insidermonkey.com,Anlaysts Bullish on Broadcom ( AVGO ) Despite...,"(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({..."
7,GOOGL,finance.yahoo.com,Investors Are Bullish on Alphabet Stock - Pili...,ERROR: HTTP Error 429: Too Many Requests
8,GOOGL,finance.yahoo.com,Is Alphabet ( GOOGL ) One of the Best Long Ter...,ERROR: HTTP Error 429: Too Many Requests
9,GOOGL,insidermonkey.com,Alphabet ( GOOGL ) Sees Optimistic Coverage Fr...,"(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({..."


In [28]:
article_text_full_df

,ticker,domain,seendate,url,title,raw_text,clean_text,selected_text
0,AAPL,insidermonkey.com,20260315T091500Z,https://www.insidermonkey.com/blog/apple-aapl-...,Apple ( AAPL ) Postpones Its Smart Home Displa...,"(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({...","(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({...","(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({..."
1,AMZN,finance.yahoo.com,20260403T044500Z,https://finance.yahoo.com/markets/stocks/artic...,Is Amazon . com ( AMZN ) One of the Best AI In...,,,ERROR: HTTP Error 429: Too Many Requests
2,AMZN,insidermonkey.com,20260305T011500Z,https://www.insidermonkey.com/blog/amazon-amzn...,Amazon ( AMZN ) Buys George Washington Univers...,"(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({...","(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({...","(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({..."
3,AMZN,insidermonkey.com,20260309T210000Z,https://www.insidermonkey.com/blog/bofa-mainta...,"BofA Maintains Buy on Amazon . com , Inc . ( A...","(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({...","(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({...","(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({..."
4,AVGO,finance.yahoo.com,20260124T010000Z,https://finance.yahoo.com/news/analysts-remain...,Analysts Remain Bullish on Broadcom ( AVGO ) W...,,,ERROR: HTTP Error 429: Too Many Requests
5,AVGO,finance.yahoo.com,20260225T084500Z,https://finance.yahoo.com/news/citi-lowers-pt-...,Citi Lowers PT on Broadcom Inc . ( AVGO ) Stock,,,ERROR: HTTP Error 429: Too Many Requests
6,AVGO,insidermonkey.com,20260401T173000Z,https://www.insidermonkey.com/blog/anlaysts-bu...,Anlaysts Bullish on Broadcom ( AVGO ) Despite...,"(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({...","(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({...","(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({..."
7,GOOGL,finance.yahoo.com,20260306T060000Z,https://finance.yahoo.com/news/investors-bulli...,Investors Are Bullish on Alphabet Stock - Pili...,,,ERROR: HTTP Error 429: Too Many Requests
8,GOOGL,finance.yahoo.com,20260412T213000Z,https://finance.yahoo.com/markets/stocks/artic...,Is Alphabet ( GOOGL ) One of the Best Long Ter...,,,ERROR: HTTP Error 429: Too Many Requests
9,GOOGL,insidermonkey.com,20260317T074500Z,https://www.insidermonkey.com/blog/alphabet-go...,Alphabet ( GOOGL ) Sees Optimistic Coverage Fr...,"(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({...","(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({...","(function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({..."


## 9. Source Usefulness Review

This final table turns the source-review exercise into a product-facing judgment: what each source is good for, where it is weak, and whether it belongs in the structured layer or the vector layer.


In [27]:
source_review = pd.DataFrame([
    {
        'source_name': 'SEC filing metadata',
        'artifact': 'latest_reporting_filings',
        'storage_layer': 'structured',
        'best_for': 'Official filing inventory, dates, forms, and primary document URLs',
        'weakness': 'Not insight by itself; mostly pointers and metadata',
        'product_value': 'Very high',
    },
    {
        'source_name': 'SEC company facts review',
        'artifact': 'company_facts_review',
        'storage_layer': 'structured',
        'best_for': 'Availability check and breadth of official company facts',
        'weakness': 'Summary only, not the full fact history',
        'product_value': 'High',
    },
    {
        'source_name': 'SEC company facts full dataset',
        'artifact': 'company_facts_full_df',
        'storage_layer': 'structured',
        'best_for': 'Official fact history by tag, unit, filing date, and fiscal period',
        'weakness': 'Still very wide and detailed without tag modeling',
        'product_value': 'Very high',
    },
    {
        'source_name': 'SEC fact tag dictionary',
        'artifact': 'fact_tag_dictionary_df',
        'storage_layer': 'structured',
        'best_for': 'Making the ~2000 fact tags understandable through business-friendly categories',
        'weakness': 'Rule-based mapping will still need refinement over time',
        'product_value': 'Very high',
    },
    {
        'source_name': 'SEC document inventory review',
        'artifact': 'sec_document_inventory_df',
        'storage_layer': 'structured',
        'best_for': 'Understanding which filing document was fetched and from where',
        'weakness': 'Still document metadata, not narrative insight',
        'product_value': 'High',
    },
    {
        'source_name': 'SEC document text full dataset',
        'artifact': 'sec_document_text_full_df',
        'storage_layer': 'vector',
        'best_for': 'Reviewing full cleaned filing text and extracted narrative sections',
        'weakness': 'Limited to the inflated filing sample, not every filing in the universe',
        'product_value': 'Very high',
    },
    {
        'source_name': 'SEC cleaning review',
        'artifact': 'sec_cleaning_review_df',
        'storage_layer': 'structured',
        'best_for': 'Judging whether filing text becomes readable after cleanup',
        'weakness': 'Does not yet isolate the best narrative section by itself',
        'product_value': 'High',
    },
    {
        'source_name': 'SEC section preview review',
        'artifact': 'sec_section_preview_df',
        'storage_layer': 'vector',
        'best_for': 'Risk factors, business narrative, MD&A, and explainability review',
        'weakness': 'Section extraction is heuristic and may still miss some filings',
        'product_value': 'Very high',
    },
    {
        'source_name': 'FRED macro data',
        'artifact': 'fred_df',
        'storage_layer': 'structured',
        'best_for': 'Economic weather and macro regime context',
        'weakness': 'Not company-specific',
        'product_value': 'High',
    },
    {
        'source_name': 'FMP profiles',
        'artifact': 'fmp_profiles',
        'storage_layer': 'structured',
        'best_for': 'Sector, industry, market cap, and instrument metadata',
        'weakness': 'Convenience enrichment, not source-of-truth',
        'product_value': 'Medium to high',
    },
    {
        'source_name': 'GDELT article metadata',
        'artifact': 'gdelt_articles',
        'storage_layer': 'structured',
        'best_for': 'Narrative discovery and URL inventory',
        'weakness': 'Metadata alone is shallow and noisy',
        'product_value': 'Medium',
    },
    {
        'source_name': 'GDELT fetch log',
        'artifact': 'gdelt_fetch_log_df',
        'storage_layer': 'structured',
        'best_for': 'Auditing which ticker requests hit cache, succeeded fresh, or were rate limited',
        'weakness': 'Operational visibility only, not market insight',
        'product_value': 'Medium',
    },
    {
        'source_name': 'Inflated news article review',
        'artifact': 'article_review_df',
        'storage_layer': 'structured',
        'best_for': 'Quick quality check for fetched article URLs and text usefulness',
        'weakness': 'Preview-only summary of the inflated article sample',
        'product_value': 'Medium',
    },
    {
        'source_name': 'Inflated news article text full dataset',
        'artifact': 'article_text_full_df',
        'storage_layer': 'vector',
        'best_for': 'Reviewing cleaned article text for themes, event risk, and market context',
        'weakness': 'Limited to the inflated article sample and source HTML quality',
        'product_value': 'Medium to high',
    },
])
source_review.to_csv(REVIEW_DIR / 'source_usefulness_review.csv', index=False)
source_review

,source_name,artifact,storage_layer,best_for,weakness,product_value
0,SEC filing metadata,latest_reporting_filings,structured,"Official filing inventory, dates, forms, and p...",Not insight by itself; mostly pointers and met...,Very high
1,SEC company facts review,company_facts_review,structured,Availability check and breadth of official com...,"Summary only, not the full fact history",High
2,SEC company facts full dataset,company_facts_full_df,structured,"Official fact history by tag, unit, filing dat...",Still very wide and detailed without tag modeling,Very high
3,SEC fact tag dictionary,fact_tag_dictionary_df,structured,Making the ~2000 fact tags understandable thro...,Rule-based mapping will still need refinement ...,Very high
4,SEC document inventory review,sec_document_inventory_df,structured,Understanding which filing document was fetche...,"Still document metadata, not narrative insight",High
5,SEC document text full dataset,sec_document_text_full_df,vector,Reviewing full cleaned filing text and extract...,"Limited to the inflated filing sample, not eve...",Very high
6,SEC cleaning review,sec_cleaning_review_df,structured,Judging whether filing text becomes readable a...,Does not yet isolate the best narrative sectio...,High
7,SEC section preview review,sec_section_preview_df,vector,"Risk factors, business narrative, MD&A, and ex...",Section extraction is heuristic and may still ...,Very high
8,FRED macro data,fred_df,structured,Economic weather and macro regime context,Not company-specific,High
9,FMP profiles,fmp_profiles,structured,"Sector, industry, market cap, and instrument m...","Convenience enrichment, not source-of-truth",Medium to high
